In [42]:
import pandas as pd

In [43]:
df = pd.read_csv("scripts_all.csv")

In [45]:
df.tail()

,movie,dialogue
0,Avengers,"[{'name': 'FURY', 'text': 'How bad is it?'}, {..."
1,Avengers_Endgame,"[{'name': 'TONY', 'text': '(offering his hand)..."
2,IRON MAN,"[{'name': 'TONY', 'text': 'Oh, I get it. You ..."
3,THOR,"[{'name': 'THOR', 'text': ""Written by Ashley M..."
4,THOR RAGNAROK,"[{'name': 'THOR', 'text': ""Now I know what you..."


In [46]:
df.shape

(5, 2)

In [47]:
df.iloc[-1]["dialogue"]

'[{\'name\': \'THOR\', \'text\': "Now I know what youâ€™re thinking. Oh no!  Thorâ€™s in a cage.  How did this happen? (then:) Well, sometimes you have to get captured just to get a straight answer out of somebody. Itâ€™s a long story but basically I\'m a bit of a hero. See, I spent some time on earth, fought some robots, saved the planet a couple of times. Then I went searching through the cosmos for some magic, colorful Infinity Stone things... didnâ€™t find any. Thatâ€™s when I came across a path of death and destruction which led me all the way here into this cage... where I met you. Reveal that heâ€™s talking to a weird alien skeleton in chains. THOR (CONTâ€™D) How much longer do you think weâ€™ll be here? NOISE.  Sounds of metal gears TURNING. Suddenly the bottom DROPS OUT!  Thor PLUMMETS until - SHLINK! Chains CINCH and abruptly halt his descent.  Thor dangles six feet off the ground.  Pull back to reveal we are in- INT. SURTURâ€™S LAIR - CONTINUOUS 3 3 A cavernous space.  The w

In [49]:
import ast

dialouge = {}

for index, row in df.iterrows():

    dialogue_list = ast.literal_eval(row["dialogue"])

    for item in dialogue_list:

        if item["name"] in dialouge:
            dialouge[item["name"]] += item["text"]
        else:
            dialouge[item["name"]] = item["text"] + " "

In [50]:
len(dialouge)

12

In [51]:
new_df = pd.DataFrame()
new_df['character'] = dialouge.keys()
new_df['words'] = dialouge.values()

In [52]:
new_df

,character,words
0,FURY,How bad is it? NASA didnâ€™t authorize Selvig ...
1,AGENT COULSON,"Thatâ€™s the problem, sir. We donâ€™t know. 5 ..."
2,MARIA HILL,It just turned itself on? (CONTINUED) CONTINUE...
3,SELVIG,"Director. The Tesseract is misbehaving.No, itâ..."
4,BARTON,I see better from a distance. No oneâ€™s come ...
5,LOKI,You have heart. (CONTINUED) CONTINUED: (2) 7. ...
6,NATASHA,(in Russian) I know how you wanted this evenin...
7,BANNER,(in Hindi) Slow down. YOUNG GIRL (in Hindi) My...
8,STEVE,"(hitting the bag) Slept for seventy years, Sir..."
9,TONY,(into headset) Stark Towerâ€™s about to become...


In [53]:
new_df["word_count"] = new_df["words"].str.split().str.len()

In [54]:
new_df = new_df.sort_values("word_count", ascending=False)

In [55]:
top30 = new_df.head(30)

print(top30[["character", "word_count"]])

        character  word_count
11           THOR       41857
9            TONY       41231
5            LOKI       12489
8           STEVE        8690
3          SELVIG        6638
6         NATASHA        6252
7          BANNER        3934
0            FURY        3381
4          BARTON        1959
1   AGENT COULSON        1415
2      MARIA HILL         938
10         JARVIS         795


In [56]:
top30.head()

,character,words,word_count
11,THOR,Where is the Tesseract? Do I look to be in a g...,41857
9,TONY,(into headset) Stark Towerâ€™s about to become...,41231
5,LOKI,You have heart. (CONTINUED) CONTINUED: (2) 7. ...,12489
8,STEVE,"(hitting the bag) Slept for seventy years, Sir...",8690
3,SELVIG,"Director. The Tesseract is misbehaving.No, itâ...",6638


In [57]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(stop_words='english')

In [58]:
embeddings = cv.fit_transform(top30['words']).toarray()

In [59]:
embeddings.shape

(12, 9961)

In [60]:
embeddings = embeddings.astype('float64')

In [61]:
from sklearn.manifold import TSNE

In [63]:
tsne = TSNE(
    n_components=2,
    perplexity=5,
    verbose=1,
    random_state=123
)

z = tsne.fit_transform(embeddings)

[t-SNE] Computing 11 nearest neighbors...
[t-SNE] Indexed 12 samples in 0.001s...
[t-SNE] Computed neighbors for 12 samples in 2.181s...
[t-SNE] Computed conditional probabilities for sample 12 / 12
[t-SNE] Mean sigma: 74.777915
[t-SNE] KL divergence after 250 iterations with early exaggeration: 55.963657
[t-SNE] KL divergence after 1000 iterations: 0.254726


In [64]:
z.shape

(12, 2)

In [65]:
top30["x"] =z[:, 0]
top30["y"] =z[:, 1]


In [66]:
top30

,character,words,word_count,x,y
11,THOR,Where is the Tesseract? Do I look to be in a g...,41857,18.195728,-13.278081
9,TONY,(into headset) Stark Towerâ€™s about to become...,41231,-15.400810,-20.787737
5,LOKI,You have heart. (CONTINUED) CONTINUED: (2) 7. ...,12489,-17.414171,7.172830
8,STEVE,"(hitting the bag) Slept for seventy years, Sir...",8690,-4.850488,-11.809994
3,SELVIG,"Director. The Tesseract is misbehaving.No, itâ...",6638,25.046864,13.716648
6,NATASHA,(in Russian) I know how you wanted this evenin...,6252,0.395468,2.536785
7,BANNER,(in Hindi) Slow down. YOUNG GIRL (in Hindi) My...,3934,-6.166217,21.388577
0,FURY,How bad is it? NASA didnâ€™t authorize Selvig ...,3381,14.488001,42.512352
4,BARTON,I see better from a distance. No oneâ€™s come ...,1959,9.472191,19.607441
1,AGENT COULSON,"Thatâ€™s the problem, sir. We donâ€™t know. 5 ...",1415,18.020016,29.737747


In [67]:
import plotly.express as px
fig = px.scatter(top30.head(25), x="x", y="y", color="character")
fig.show()

In [68]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(embeddings)

print(similarity.shape)

(12, 12)


In [69]:
similarity

array([[1.        , 0.30790324, 0.83329491, 0.46627096, 0.58942552,
        0.47122112, 0.58602248, 0.35722426, 0.42127069, 0.30465767,
        0.2977597 , 0.27329538],
       [0.30790324, 1.        , 0.28689146, 0.51383701, 0.20228506,
        0.49187213, 0.38238096, 0.39527398, 0.35421074, 0.38540037,
        0.28213248, 0.37825397],
       [0.83329491, 0.28689146, 1.        , 0.35859863, 0.51911698,
        0.41731009, 0.49193518, 0.43502999, 0.46499281, 0.32232879,
        0.28488065, 0.27597537],
       [0.46627096, 0.51383701, 0.35859863, 1.        , 0.24775075,
        0.71445891, 0.45055091, 0.39194911, 0.34169666, 0.3294945 ,
        0.27778381, 0.26012063],
       [0.58942552, 0.20228506, 0.51911698, 0.24775075, 1.        ,
        0.25547523, 0.27476375, 0.25442481, 0.26485539, 0.26147189,
        0.1946356 , 0.19810687],
       [0.47122112, 0.49187213, 0.41731009, 0.71445891, 0.25547523,
        1.        , 0.5367983 , 0.49252716, 0.53851915, 0.43480512,
        0.35691539,

In [70]:
print(top30["character"].values)

<StringArray>
[         'THOR',          'TONY',          'LOKI',         'STEVE',
        'SELVIG',       'NATASHA',        'BANNER',          'FURY',
        'BARTON', 'AGENT COULSON',    'MARIA HILL',        'JARVIS']
Length: 12, dtype: str


In [71]:
character = "TONY"

index = new_df.index[new_df["character"] == character][0]

scores = similarity[index]

print(scores)

[0.30465767 0.38540037 0.32232879 0.3294945  0.26147189 0.43480512
 0.43441462 0.62936539 0.46496724 1.         0.45849249 0.33778042]


In [72]:
similar_indices = scores.argsort()[::-1]

print(similar_indices)

[ 9  7  8 10  5  6  1 11  3  2  0  4]


In [73]:
top_indices = similar_indices[1:6]

for i in top_indices:
    print(new_df.iloc[i]["character"], scores[i])

FURY 0.6293653886296156
BARTON 0.4649672378982549
MARIA HILL 0.4584924875311344
NATASHA 0.4348051210052964
BANNER 0.4344146165874033


In [74]:
def find_similar(character, similarity, df, n=5):

    index = df.index[df["character"] == character][0]

    scores = similarity[index]

    similar_indices = scores.argsort()[::-1]

    top_indices = similar_indices[1:n+1]

    for i in top_indices:
        print(df.iloc[i]["character"], scores[i])

In [75]:
find_similar("TONY", similarity, new_df)

FURY 0.6293653886296156
BARTON 0.4649672378982549
MARIA HILL 0.4584924875311344
NATASHA 0.4348051210052964
BANNER 0.4344146165874033


In [76]:
find_similar("STEVE", similarity, new_df)

FURY 0.5825386758959901
NATASHA 0.5385191465925172
MARIA HILL 0.497205062228617
JARVIS 0.4913227354580314
LOKI 0.46499280778486995


In [79]:
def find_similar(character, similarity, df, n=5):

    index = df.index[df["character"] == character][0]

    scores = similarity[index]

    similar_indices = scores.argsort()[::-1]

    top_indices = similar_indices[1:n+1]

    results = []

    for i in top_indices:
        results.append({
    "character": df.iloc[i]["character"],
    "score": round(scores[i] * 100, 2)
})

    return results

In [80]:
results = find_similar("TONY", similarity, new_df)

print(results)

[{'character': 'FURY', 'score': np.float64(62.94)}, {'character': 'BARTON', 'score': np.float64(46.5)}, {'character': 'MARIA HILL', 'score': np.float64(45.85)}, {'character': 'NATASHA', 'score': np.float64(43.48)}, {'character': 'BANNER', 'score': np.float64(43.44)}]


In [81]:
characters = new_df["character"].tolist()

print(characters)

['THOR', 'TONY', 'LOKI', 'STEVE', 'SELVIG', 'NATASHA', 'BANNER', 'FURY', 'BARTON', 'AGENT COULSON', 'MARIA HILL', 'JARVIS']


In [82]:
pip install streamlit

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
   ---------------------------------------- 0.0/10.5 MB ? eta -:--:--
   --------- ------------------------------ 2.6/10.5 MB 16.2 MB/s eta 0:00:01
   ---------------------- ----------------- 6.0/10.5 MB 15.7 MB/s eta 0:00:01
   ------------------------------------ --- 9.7/10.5 MB 16.6 MB/s eta 0:00:01
   ---------------------------------------- 10.5/10.5 MB 14.6 MB/s  0:00:00
   ---------------------------------------- 0.0/797.6 kB ? eta -:--:--
   ---------------------------------------- 797.6/797.6 kB 11.2 MB/s  0:00:00
   ---------------------------------------- 0.0/28.0 MB ? eta -:--:--
   --- ------------------------------------ 2.6/28.0 MB 8.3 MB/s eta 0:00:04
   --------- ------------------------------ 6.8/28.0 MB 11.4 MB/s eta 0:00:02
   ------------- -------------------------- 9.4/28.0 MB 11.6 MB/s eta 0:00:02
   ----------------- ---------------------- 12.1/28.0 MB 11.7 MB/s eta 0:00:02
   --------------------- -


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [83]:
similarity = cosine_similarity(embeddings)

In [85]:
new_df.to_csv("characters.csv", index=False)

In [86]:
import numpy as np

np.save("similarity.npy", similarity)